# Gear 2.2 · Z1 · quiet-spread normalization transferability (manifest v2, scratch)

Read-only view of `research/output/gear22_z1_quiet_normalization/v2/`.
No canon / VARIATION / HYPER / Trade_Lat / fee retune. No PnL.

Report: `research/gear22_z1_quiet_normalization.md`  
Atlas: `research/output/gear22_z1_quiet_normalization/v2/atlas.html`  
v1 pilot artifacts are frozen under `.../gear22_z1_quiet_normalization/v1/`.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.width', 220)
V = Path('research/output/gear22_z1_quiet_normalization/v2')
V1 = Path('research/output/gear22_z1_quiet_normalization/v1')

verdict = json.loads((V / 'verdict.json').read_text())
print(verdict['verdict'], '| sensitivity incl. tick-limited:', verdict['sensitivity_including_tick_limited'])
print('cond 14.1-1 met:', verdict['cond1_met'])
print('cond 14.1-2 met (observed):', {d: verdict['observed_arm'][d]['met'] for d in ('long', 'short')})
print('cond 14.1-2 met (resolution-aware):', {d: verdict['resolution_aware_arm'][d]['met'] for d in ('long', 'short')})
print('LOCO global pass rates:', json.dumps(verdict['loco_global_pass_rates'], indent=2))

FileNotFoundError: [Errno 2] No such file or directory: 'research/output/gear22_z1_quiet_normalization/v2/verdict.json'

## Frozen manifest v2

In [ ]:
meta = json.loads((V / 'manifest_frozen_meta.json').read_text())
print(json.dumps({k: meta[k] for k in ('n_declared', 'n_primary', 'rejected', 'windows_per_coin',
                                       'n_quiet_regimes_primary', 'n_calendar_clusters_primary',
                                       'calendar_cluster_sizes_primary')}, indent=2))
man = pd.read_csv(V / 'manifest_frozen.csv')
man[['quiet_id', 'liquidity_class_pre', 'quiet_regime_id', 'calendar_cluster_id',
     'n_ticks_cal', 'n_ticks_eval', 'n_miss', 'unk_frac', 'primary', 'rejection_reason']]

## Window statistics, tails and discretisation

In [ ]:
st = pd.read_csv(V / 'window_stats.csv')
print(st.groupby(['direction', 'normalization_status']).size().to_string())
print()
print('median |delta_F| =', round(st.delta_F.abs().median(), 3),
      '| frac > 0.5 sigma =', round((st.delta_F.abs() > 0.5).mean(), 3),
      '| max =', round(st.delta_F.abs().max(), 3))
print('frac scale change > 25% =', round((st.L_sigma.abs() > np.log(1.25)).mean(), 3))
st[['quiet_id', 'direction', 'F0', 'sigma0', 'F1', 'sigma1', 'delta_F', 'L_sigma',
    'kappa', 'kappa_eval', 'n_occupied_levels', 'a_max',
    'normalization_status', 'local_status']].round(4)

In [ ]:
ok = st[st.normalization_status == 'ok']
rows = []
for rep, label in (('rc', 'r_causal'), ('rl', 'r_local'), ('zp', 'z_plus_causal')):
    for p in (95, 97, 99):
        rows.append({'representation': label, 'p': p,
                     'median_observed': ok[f'{rep}_Q{p}'].median(),
                     'median_resolution_aware': ok[f'{rep}_Q{p}_ra'].median()})
print(pd.DataFrame(rows).round(3).to_string(index=False))
print()
dc = pd.read_csv(V / 'discretisation.csv')
print('kappa quartiles (usable):',
      [round(x, 3) for x in dc[dc.normalization_status == 'ok'].kappa.quantile([0, .25, .5, .75, 1]).tolist()])
print('kappa median per coin:')
print(dc.groupby('base_coin').kappa.median().sort_values().round(3).to_string())

## Transfer: within-coin and leave-one-coin-out

In [ ]:
tr = pd.read_csv(V / 'within_coin_transfer.csv')
print('within-coin pass rate')
print(tr.groupby(['representation', 'direction'])[['pass_observed', 'pass_resolution_aware']].mean().round(3).to_string())
print()
lo = pd.read_csv(V / 'loco_calibration.csv')
g = lo[(lo.scope == 'global') & (~lo.descriptive_only)]
print('LOCO global pass rate')
print(g.groupby(['representation', 'direction'])[['pass_observed', 'pass_resolution_aware']].mean().round(3).to_string())
print()
print('LOCO pass rate by kappa bin (r_causal, global)')
gr = g[g.representation == 'rc'].copy()
gr['kbin'] = pd.cut(gr.kappa, [0, 0.05, 0.15, 0.35, 0.6, 1.0])
print(gr.groupby('kbin', observed=True)[['pass_observed', 'pass_resolution_aware']].agg(['mean', 'size']).round(3).to_string())
print()
print('violation direction (r_causal, global, observed)')
print(gr.groupby(['direction', 'direction_of_violation']).size().unstack(fill_value=0).to_string())

In [ ]:
# sensitivity: raw time-weighted pooling instead of equal weight per coin
tw = pd.read_csv(V / 'loco_calibration_time_weighted.csv')
key = ['scope', 'base_coin', 'direction', 'representation', 'quiet_id', 'p']
m = lo.merge(tw, on=key, suffixes=('_eq', '_tw'))
gm = m[(m.scope == 'global') & (m.representation == 'rc') & (~m.descriptive_only_eq)]
print('rows', len(gm),
      '| pooled-threshold rows that differ', int((gm.q_pool_eq != gm.q_pool_tw).sum()),
      '| pass flips', int((gm.pass_observed_eq != gm.pass_observed_tw).sum()))
print('pass rate equal-coin', round(gm.pass_observed_eq.mean(), 4),
      '| time-weighted', round(gm.pass_observed_tw.mean(), 4))

## W1 distances, variance decomposition, ICC

In [ ]:
di = pd.read_csv(V / 'distances.csv')
print('median W1 by representation and pair type')
print(di.groupby(['representation', 'direction', 'same_coin'])[['w1_raw', 'w1_res_matched']].median().round(3).to_string())
print()
print(pd.read_csv(V / 'variance_decomposition.csv')[
    ['direction', 'metric', 'arm', 'n', 'frac_class', 'frac_coin', 'frac_window']].round(3).to_string(index=False))

In [ ]:
ic = pd.read_csv(V / 'icc.csv')
print('ICC, cluster bootstrap over coins, r_causal')
print(ic[(ic.conditioning == 'coin') & ic.metric.str.startswith('rc_Q')][
    ['direction', 'metric', 'arm', 'icc', 'lcb95', 'ucb95', 'var_coin', 'var_window']].round(3).to_string(index=False))
print()
print('sensitivity: cluster bootstrap over calendar_cluster_id')
icc = pd.read_csv(V / 'icc_calendar_cluster.csv')
print(icc[icc.conditioning == 'coin'][['direction', 'metric', 'icc', 'lcb95', 'ucb95']].round(3).to_string(index=False))

## v2 against the v1 pilot

In [ ]:
v1v = json.loads((V1 / 'verdict.json').read_text())
v1m = json.loads((V1 / 'manifest_frozen_meta.json').read_text())
cmp = pd.DataFrame([
    {'metric': 'verdict', 'v1': v1v['verdict'], 'v2': verdict['verdict']},
    {'metric': 'primary windows', 'v1': v1m['n_primary'], 'v2': meta['n_primary']},
    {'metric': 'rejected windows', 'v1': len(v1m['rejected']), 'v2': len(meta['rejected'])},
    {'metric': 'quiet regimes', 'v1': v1v['n_quiet_regimes'], 'v2': verdict['n_quiet_regimes']},
    {'metric': 'coins with >=2 windows', 'v1': v1v['n_coins_with_ge2_windows'], 'v2': verdict['n_coins_with_ge2_windows']},
    {'metric': 'tick_resolution_limited', 'v1': v1v['n_windows_tick_resolution_limited'], 'v2': verdict['n_windows_tick_resolution_limited']},
    {'metric': 'LOCO long obs', 'v1': round(v1v['loco_global_pass_rates']['long']['observed'], 3),
     'v2': round(verdict['loco_global_pass_rates']['long']['observed'], 3)},
    {'metric': 'LOCO long ra', 'v1': round(v1v['loco_global_pass_rates']['long']['resolution_aware'], 3),
     'v2': round(verdict['loco_global_pass_rates']['long']['resolution_aware'], 3)},
    {'metric': 'LOCO short obs', 'v1': round(v1v['loco_global_pass_rates']['short']['observed'], 3),
     'v2': round(verdict['loco_global_pass_rates']['short']['observed'], 3)},
    {'metric': 'LOCO short ra', 'v1': round(v1v['loco_global_pass_rates']['short']['resolution_aware'], 3),
     'v2': round(verdict['loco_global_pass_rates']['short']['resolution_aware'], 3)},
])
print(cmp.to_string(index=False))
print()
print('v1 offenders long:', [o['coin'] for o in v1v['observed_arm']['long']['offenders']])
print('v2 offenders long:', [o['coin'] for o in verdict['observed_arm']['long']['offenders']])
print('v1 offenders short:', [o['coin'] for o in v1v['observed_arm']['short']['offenders']])
print('v2 offenders short:', [o['coin'] for o in verdict['observed_arm']['short']['offenders']])

## The degenerate sigma1 window disclosed in section 12 of the report

The frozen contract defines `tick_resolution_limited` on `kappa = delta_s / sigma0` only, so it does
not screen a degenerate evaluation-half scale. `BIO_260815_00` short passes the contract
(`kappa = 0.68`) while `kappa_eval = 22.7`. The contract was not changed; the influence is bounded here.

In [ ]:
bad = 'BIO_260815_00'
print(dc[dc.quiet_id == bad][['direction', 'sigma0', 'sigma1', 'delta_s', 'kappa', 'kappa_eval',
                             'mad_status', 'normalization_status', 'local_status']].to_string(index=False))
print()
s = st[st.direction == 'short']
for label, d in (('all', s), ('excluding it', s[s.quiet_id != bad])):
    print(label, 'median rl_Q99', round(d.rl_Q99.median(), 3), '| max rl_Q99', round(d.rl_Q99.max(), 2))
d2 = di[(di.representation == 'rl') & (di.a != bad) & (di.b != bad)]
print()
print('W1 r_local excluding it')
print(d2.groupby(['direction', 'same_coin'])[['w1_raw', 'w1_res_matched']].median().round(3).to_string())